In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix

# Reading test, validation, and training data from files

df_test = pd.read_csv('/workspaces/churn-analysis/datasets/test.csv')
df_val = pd.read_csv('/workspaces/churn-analysis/datasets/val.csv')
df_train = pd.read_csv('/workspaces/churn-analysis/datasets/val.csv')

y_test = df_test['churn']
y_val = df_val['churn']
y_train = df_train['churn']

# First rows of the dataframes
print("Test Data:")
print(df_test.head(3))
      

df_test.drop(columns=['churn'], inplace=True)
df_val.drop(columns=['churn'], inplace=True)
df_train.drop(columns=['churn'], inplace=True) 

y_val = (y_val=='yes').astype(int)
y_test = (y_test=='yes').astype(int)



Test Data:
   gender  seniorcitizen partner dependents  tenure phoneservice  \
0  female              0      no         no      41          yes   
1  female              1      no         no      66          yes   
2  female              0      no         no      12          yes   

  multiplelines internetservice onlinesecurity onlinebackup deviceprotection  \
0            no             dsl            yes           no              yes   
1           yes     fiber_optic            yes           no               no   
2            no             dsl             no           no               no   

  techsupport streamingtv streamingmovies        contract paperlessbilling  \
0         yes         yes             yes        one_year              yes   
1          no         yes             yes        two_year              yes   
2          no          no              no  month-to-month              yes   

               paymentmethod  monthlycharges  totalcharges churn  
0  bank_transfe

In [59]:
train_dicts = df_train.to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

test_dicts = df_test.to_dict(orient='records')
X_test = dv.transform(test_dicts)

dv.feature_names_

['contract=month-to-month',
 'contract=one_year',
 'contract=two_year',
 'dependents=no',
 'dependents=yes',
 'deviceprotection=no',
 'deviceprotection=no_internet_service',
 'deviceprotection=yes',
 'gender=female',
 'gender=male',
 'internetservice=dsl',
 'internetservice=fiber_optic',
 'internetservice=no',
 'monthlycharges',
 'multiplelines=no',
 'multiplelines=no_phone_service',
 'multiplelines=yes',
 'onlinebackup=no',
 'onlinebackup=no_internet_service',
 'onlinebackup=yes',
 'onlinesecurity=no',
 'onlinesecurity=no_internet_service',
 'onlinesecurity=yes',
 'paperlessbilling=no',
 'paperlessbilling=yes',
 'partner=no',
 'partner=yes',
 'paymentmethod=bank_transfer_(automatic)',
 'paymentmethod=credit_card_(automatic)',
 'paymentmethod=electronic_check',
 'paymentmethod=mailed_check',
 'phoneservice=no',
 'phoneservice=yes',
 'seniorcitizen',
 'streamingmovies=no',
 'streamingmovies=no_internet_service',
 'streamingmovies=yes',
 'streamingtv=no',
 'streamingtv=no_internet_servic

In [60]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train) 

for k in np.arange(0, 1.0, 0.05):
    y_pred = (model.predict_proba(X_val)[:, 1] >= k).astype(int)
    print(round(k, 2), round((y_pred == y_val).mean(), 2))
    print(pd.crosstab(y_val, y_pred))


0.0 0.27
col_0     1
churn      
0      1023
1       386
0.05 0.52
col_0    0    1
churn          
0      361  662
1        8  378
0.1 0.62
col_0    0    1
churn          
0      508  515
1       21  365
0.15 0.68
col_0    0    1
churn          
0      614  409
1       37  349
0.2 0.73
col_0    0    1
churn          
0      688  335
1       49  337
0.25 0.76
col_0    0    1
churn          
0      750  273
1       66  320
0.3 0.79
col_0    0    1
churn          
0      798  225
1       77  309
0.35 0.79
col_0    0    1
churn          
0      835  188
1      101  285
0.4 0.8
col_0    0    1
churn          
0      866  157
1      128  258
0.45 0.8
col_0    0    1
churn          
0      894  129
1      147  239
0.5 0.81
col_0    0    1
churn          
0      923  100
1      168  218
0.55 0.81
col_0    0    1
churn          
0      946   77
1      186  200
0.6 0.81
col_0    0    1
churn          
0      966   57
1      205  181
0.65 0.8
col_0    0    1
churn          
0      982   41
1     

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [61]:
y_pred_test = (model.predict_proba(X_test)[:, 1] >= 0.4).astype(int)

print(round((y_pred_test == y_test).mean(), 2))
print(pd.crosstab(y_test, y_pred_test))

0.79
col_0    0    1
churn          
0      867  194
1      104  244
